# Integración de datos en MySQL

## Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import mysql.connector
import os
from dotenv import load_dotenv

## Lectura CSV

In [3]:
# Leemos los archivos CSV que generamos cuando leimos los Jsons creando los dataframes

df_contenidos = pd.read_csv("../data/csv/df_contenidos.csv", encoding="utf-8")
df_compartir_tiempo = pd.read_csv("../data/csv/df_compartir_tiempo.csv", encoding="utf-8")
df_fake = pd.read_csv("../data/csv/df_fake.csv", encoding="utf-8")
df_info = pd.read_csv("../data/csv/df_info.csv", encoding="utf-8")
df_fuente = pd.read_csv("../data/csv/df_fuente.csv", encoding="utf-8")
df_metadatos = pd.read_csv("../data/csv/df_metadatos.csv", encoding="utf-8")
df_pais_idioma = pd.read_csv("../data/csv/df_pais_idioma.csv", encoding="utf-8")
df_tiempo = pd.read_csv("../data/csv/df_tiempo.csv", encoding="utf-8")
df_tipo = pd.read_csv("../data/csv/df_tipo.csv", encoding="utf-8")

## Ajustes y unión de DataFrames

In [5]:
# Cambiamos los nombres de las columnas 'id' para que cada df tenga una columna con diferente nombre pero que represente el id de la noticia 

df_fake.rename(columns={'id': 'id_fake'},inplace=True)
df_info.rename(columns={'id': 'id_info'},inplace=True)
df_pais_idioma.rename(columns={'id': 'id_pais_idioma'},inplace=True)
df_tiempo.rename(columns={'id': 'id_tiempo'},inplace=True)
df_tipo.rename(columns={'id': 'id_tipo'},inplace=True)
df_fuente.rename(columns={'id': 'id_fuente'},inplace=True)

In [6]:
# Unimos los DataFrames en uno solo

df_completo = df_contenidos.merge(df_compartir_tiempo, left_on="id_contenido", right_on="id_compartir", how="outer") \
              .merge(df_fake, left_on="id_contenido", right_on="id_fake", how="outer") \
              .merge(df_info, left_on="id_contenido", right_on="id_info", how="outer") \
              .merge(df_metadatos, left_on="id_contenido", right_on="t_id", how="outer") \
              .merge(df_pais_idioma, left_on="id_contenido", right_on="id_pais_idioma", how="outer") \
              .merge(df_tiempo, left_on="id_contenido", right_on="id_tiempo", how="outer") \
              .merge(df_tipo, left_on="id_contenido", right_on="id_tipo", how="outer") \
              .merge(df_fuente, left_on="id_contenido", right_on="id_fuente", how="outer")

In [7]:
df_completo.drop(['id_compartir', 'id_fake','id_info','t_id','id_pais_idioma','id_tiempo','id_tipo','id_fuente'], axis=1,inplace = True)

In [8]:
# Creamos 2 columnas nuevas que reflejan los valores de las columnas 'tiempo' y 'compartir_tiempo' en formato datetime

fecha_inicio = pd.Timestamp('2022-09-01 07:00:00')# Hora de inicio

df_completo['fecha_hora_compartir'] = (fecha_inicio + pd.to_timedelta(df_completo['compartir_tiempo'], unit='h'))

df_completo['fecha_hora_tiempo'] = (fecha_inicio + pd.to_timedelta(df_completo['tiempo'], unit='h'))

In [12]:
# Funcion para codificar una columna categorica

def codificacion(columna_para_codificar) :

    # Codificamos los valores
    df_completo['codificacion'] = df_completo[f'{columna_para_codificar}'].astype('category')
    df_completo[f'{columna_para_codificar}_codificado'] = df_completo['codificacion'].cat.codes

    # leemos a que valores pertenece cada codigo
    mapping = dict(enumerate(df_completo['codificacion'].cat.categories))
    mapping_con_nulos = {-1: "NaN"}
    mapping_con_nulos.update(mapping)

    # Creamos un Df con los valores y codigos
    df_codificacion = pd.DataFrame(list(mapping_con_nulos.items()), columns=['codigo', 'valor'])
     
    
    del df_completo['codificacion']
    return df_codificacion
    

In [17]:
# Generamos los dfs que nos serviran para crear nuestras tablas fijas en SQL

df_codigos_tipo = codificacion('tipo')
df_codigos_fuente = codificacion('fuente')
df_codigos_pais = codificacion('pais')
df_codigos_autor = codificacion('autor')
df_codigos_fake = codificacion('fake')

In [19]:
# Generamos los dfs que nos serviran para crear las tablas de titulo y texto

df_titulo = df_completo[['id_contenido','titulo']]
df_texto = df_completo[['id_contenido','texto']]

In [21]:
# Generamos el df correspondiente a nuestra tabla principal de noticias en MySQL

df_noticias = df_completo[['id_contenido','fake_codificado','tipo_codificado',
                           'visitas','compartir','duracion','favorito','autor_codificado',
                           'fuente_codificado','pais_codificado','fecha_hora_tiempo','fecha_hora_compartir']].copy()

In [23]:
# Ahora hacemos unos ajustes a los datos para poder migrarlos correctamente a Mysql

df_codigos_fake = df_codigos_fake.drop(df_codigos_fake.index[0]) # Borramos una linea en el df para ajustarlo al tipo de dato SQL (bit)

df_noticias['visitas'] = pd.to_numeric(df_noticias['visitas'], errors='coerce') # Cambiamos la columna de tipo 'object' a 'numerico'
df_noticias['favorito'] = pd.to_numeric(df_noticias['favorito'], errors='coerce') # Cambiamos la columna de tipo 'object' a 'numerico'

# Conexión con Base de datos (MySQL)

In [67]:
# Conexión (los datos de usuarios, contraseñas, puertos y host estan en un archivo .env)

load_dotenv()

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASS"),
    database=os.getenv("DB_NAME")
)


# Inserción de datos en las tablas de la base de datos

In [69]:
# Creamos una funcion que inserte los datos en las tablas SQL

def insertar_datos (codigo , nombre, df, tabla):
    sql = f"INSERT INTO {tabla} ({codigo}, {nombre}) VALUES (%s, %s)"
    valores = [tuple(x) for x in df.to_numpy()]
    cursor.executemany(sql, valores)
    conn.commit()

In [71]:
# Insertamos los datos correspondientes en cada tabla fija en SQL

cursor = conn.cursor()

insertar_datos('codigo_tipo', 'nombres_tipo', df_codigos_tipo,'fija_tipo') # Insertamos datos en la tabla fija_tipo
insertar_datos('codigo_fuente', 'nombres_fuente', df_codigos_fuente, 'fija_fuente') # Insertamos datos en la tabla fija_fuente
insertar_datos('codigo_pais', 'nombres_pais', df_codigos_pais, 'fija_pais') # Insertamos datos en la tabla fija_pais
insertar_datos('codigo_autor', 'nombres_autor', df_codigos_autor, 'fija_autor') # Insertamos datos en la tabla fija_autor
insertar_datos('codigo_fake', 'nombres_fake', df_codigos_fake, 'fija_fake') # Insertamos datos en la tabla fija_fake

insertar_datos('codigo_titulo', 'titulo', df_titulo, 'titulo') # Insertamos datos en la tabla fija_titulo
insertar_datos('codigo_texto', 'texto', df_texto, 'texto') # Insertamos datos en la tabla fija_texto

In [81]:
# Insertamos los datos correspondientes a la tabla principal en SQL


sql = '''INSERT INTO noticia (t_id, fake, tipo, visitas, compartir, duracion, favorito, autor, fuente, pais, fecha_tiempo, fecha_compartir_tiempo)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)'''

valores = [
    tuple(None if pd.isna(x) else x for x in fila)
    for fila in df_noticias.to_numpy()
]

cursor.executemany(sql, valores)
conn.commit()

In [83]:
# Cerramos la conexion

cursor.close()
conn.close()